<a href="https://colab.research.google.com/github/obaidah3/rag-ecommerce-chatbot/blob/main/02_sentiment_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2. Sentiment / Emotion Classifier
Fine-tune a small Transformer (`distilbert-base-uncased`) on `dair-ai/emotion`, mapped down to 3 routing buckets: **negative / neutral / positive**.

**Design decisions (be ready to explain these):**
- The task guidelines allow RNN *or* Transformer. We use a **fine-tuned DistilBERT** because it converges in very few epochs even on modest data, which matters under a 1-day deadline — an RNN from scratch usually needs more epochs/data to reach comparable accuracy.
- `dair-ai/emotion` is **Twitter data**, not customer-support text (domain shift, noted in the brief). We mitigate this by (a) mapping to only 3 coarse buckets — easier to transfer than 6 fine-grained emotions — and (b) doing a manual qualitative check with a handful of hand-written support-style messages at the end of this notebook.
- We map: sadness/anger/fear → negative, surprise → neutral (context-dependent, treated conservatively), joy/love → positive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/chatbot_project"
MODELS_DIR = f"{PROJECT_DIR}/models"
os.makedirs(MODELS_DIR, exist_ok=True)
print("Models will be saved to:", MODELS_DIR)

Mounted at /content/drive
Models will be saved to: /content/drive/MyDrive/chatbot_project/models


In [ ]:
!pip install -q datasets transformers evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset

ds = load_dataset("dair-ai/emotion")
ds


README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

split/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.03MB            

split/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

split/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  127kB            

split/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

split/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [ ]:
# emotion label order in dair-ai/emotion: 0 sadness,1 joy,2 love,3 anger,4 fear,5 surprise
label_map = {0:0, 3:0, 4:0,   # negative
             5:1,               # neutral (surprise, ambiguous polarity)
             1:2, 2:2}          # positive

def remap(example):
    example["label3"] = label_map[example["label"]]
    return example

ds = ds.map(remap)
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {v:k for k,v in id2label.items()}


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

tokenized = ds.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column("label3", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
import datasets.config as ds_config
ds_config.TORCHVISION_AVAILABLE = False

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.077807,0.057307,0.980000,0.945784
2,0.021395,0.050373,0.978500,0.931735


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=0.10500238990783692, metrics={'train_runtime': 182.9105, 'train_samples_per_second': 174.949, 'train_steps_per_second': 5.467, 'total_flos': 529879044096000.0, 'train_loss': 0.10500238990783692, 'epoch': 2.0})

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id
)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

args = TrainingArguments(
    output_dir=f"{MODELS_DIR}/sentiment_ckpt",
    num_train_epochs=2,             # keep short — time-boxed for a 1-day deadline
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
)
trainer.train()


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.077542,0.055517,0.977000,0.936708
2,0.032431,0.050558,0.981500,0.941735


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=0.10199903190135956, metrics={'train_runtime': 178.3932, 'train_samples_per_second': 179.379, 'train_steps_per_second': 5.606, 'total_flos': 529879044096000.0, 'train_loss': 0.10199903190135956, 'epoch': 2.0})

In [ ]:
trainer.evaluate(tokenized["test"])

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.032431,0.055527,2,0.976500,0.903047


{'eval_loss': 0.055527254939079285,
 'eval_accuracy': 0.9765,
 'eval_f1_macro': 0.9030468754668745}

In [ ]:
trainer.save_model(f"{MODELS_DIR}/sentiment_model")
tokenizer.save_pretrained(f"{MODELS_DIR}/sentiment_model")
print("saved")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved


## Qualitative check on realistic customer-support-style text
(This is the domain-shift mitigation mentioned above — a small hand-labeled sanity set.)

In [ ]:
from transformers import pipeline

clf = pipeline("text-classification", model=f"{MODELS_DIR}/sentiment_model", tokenizer=f"{MODELS_DIR}/sentiment_model")

support_samples = [
    "This is the third time my order has been delayed, I am extremely frustrated!",
    "Can you tell me the status of my order #4521?",
    "Thank you so much, the refund arrived, you guys are great!",
]
for s in support_samples:
    print(s, "->", clf(s))


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

This is the third time my order has been delayed, I am extremely frustrated! -> [{'label': 'negative', 'score': 0.9992006421089172}]
Can you tell me the status of my order #4521? -> [{'label': 'negative', 'score': 0.5288427472114563}]
Thank you so much, the refund arrived, you guys are great! -> [{'label': 'positive', 'score': 0.9970314502716064}]
